In [13]:
from circular_mode_v2 import CircularMode, compute_step_gsm, compute_coupling_matrix
from get_required_modes import analyze_horn_profile_modes
import numpy as np

In [14]:
# define circular junction
#f = 20GHz
freq = 20e9
a = 0.01
b = 0.015

In [15]:
modes_col, mode_ouvertures = analyze_horn_profile_modes(r_profile=np.array([a, b]), freq_hz=freq)

=== ANALYSE MODALE (f0 = 20.0 GHz, Facteur coupure = 4.0) ===
1. COL (r = 10.00 mm) :
   -> 10 modes au total (5 TE + 5 TM)
   -> Propagatifs : 2

2. OUVERTURE (r = 15.00 mm) :
   -> 15 modes au total (8 TE + 7 TM)
   -> Propagatifs : 3


In [16]:
modes_col_ids = [f"{m['type']}{m['m']}{m['n']}" for m in modes_col]
print(modes_col_ids)
modes_a = [CircularMode(m['type'], m['m'], m['n'], a, freq) for m in modes_col]
#modes_a = [CircularMode("TE", m, 1, a, freq) for m in range(10)]

['TE11', 'TE12', 'TE13', 'TE14', 'TE15', 'TM11', 'TM12', 'TM13', 'TM14', 'TM15']


In [17]:
modes_ouverture_ids = [f"{m['type']}{m['m']}{m['n']}" for m in mode_ouvertures]
print(modes_ouverture_ids)
modes_b = [CircularMode(m['type'], m['m'], m['n'], b, freq) for m in mode_ouvertures]
#modes_b = [CircularMode("TE", m, 1, b, freq) for m in range(10)]

['TE11', 'TE12', 'TE13', 'TE14', 'TE15', 'TE16', 'TE17', 'TE18', 'TM11', 'TM12', 'TM13', 'TM14', 'TM15', 'TM16', 'TM17']


In [18]:
M = compute_coupling_matrix(modes_a, modes_b, a)
S, S11, S12, S21, S22 = compute_step_gsm(modes_a, modes_b, M)

In [19]:
w_a_p = np.zeros(S21.shape[1])
w_a_p[0] = 1.
w_b_p = S21 @ w_a_p
print(w_b_p)
print(np.abs(w_b_p))
w_a_m = S11 @ w_a_p
print(w_a_m)
print(np.abs(w_a_m))
w_b_m = np.zeros(S12.shape[1])


[ 9.21839589e-01-4.39109663e-02j  2.69698807e-01+1.18577209e-01j
  1.25955860e-02+7.86453345e-02j -1.72543258e-02-2.84798303e-02j
  7.16459092e-03-1.51154891e-02j  1.26050613e-03+2.35956618e-02j
 -3.15395227e-03-4.31256727e-03j  7.28688388e-04-1.87305424e-02j
  7.63123111e-19+2.10868815e-18j -2.13307821e-18-6.41039057e-19j
 -7.92779318e-19-4.93271388e-18j -8.01674309e-19-7.14837838e-18j
 -1.13859907e-18-5.43946699e-18j  1.10825422e-19-5.46920810e-18j
 -1.66627792e-19-8.40835107e-18j]
[9.22884826e-01 2.94615004e-01 7.96475826e-02 3.32988362e-02
 1.67275035e-02 2.36293066e-02 5.34281306e-03 1.87447114e-02
 2.24252594e-18 2.22731985e-18 4.99601494e-18 7.19319089e-18
 5.55735630e-18 5.47033084e-18 8.41000193e-18]
[ 2.04265247e-01+2.33799511e-02j  5.67318303e-02+1.49311912e-01j
 -1.00200749e-02-5.71776006e-02j  4.40999610e-03+3.37672184e-02j
 -2.55993130e-03-2.33948956e-02j  6.89546091e-18+1.57318783e-19j
  2.56406640e-18-6.77184835e-18j -2.54152640e-19-9.22974849e-18j
  1.28224104e-18-8.32

In [20]:
res_et = M.T @ (w_a_p + w_a_m) - (w_b_p + w_b_m)
print(np.linalg.norm(res_et))

1.1138930815746818e-16


In [21]:
Y1 = np.diag([m.Y for m in modes_a])
Y2 = np.diag([m.Y for m in modes_b])
M_T = M.T
res_ht = Y1 @ (w_a_p - w_a_m) - M @ Y2 @ (w_b_p - w_b_m)
print(np.linalg.norm(res_ht))

1.610699464972672e-19


In [22]:
def compute_tangent_field(mode_list: list[CircularMode], w_p, w_m, r, t, z):
    E ,H = np.zeros(3, dtype=np.complex128), np.zeros(3, dtype=np.complex128)
    for i, mode in enumerate(mode_list):
        norm = mode.norm
        e_mode_p, h_mode_p = mode.field_rtz(r, t, z)
        E += w_p[i] * e_mode_p * norm
        H += w_p[i] * h_mode_p * norm
        e_mode_m, h_mode_m = mode.field_rtz(r, t, z, dir_z=-1.0)
        E += w_m[i] * e_mode_m * norm
        H -= w_m[i] * h_mode_m * norm
    return E, H


In [25]:
z = 0.
r = 0.45 * a
t = np.radians(20.)
Ea, Ha = compute_tangent_field(modes_a, w_a_p, w_a_m, r, t, z)
Eb, Hb = compute_tangent_field(modes_b, w_b_p, w_b_m, r, t, z)
print(Ea)
print(Eb)

[-4.20672762e+00+1.44151898e+02j -3.55924162e+00+3.23550086e+02j
  2.09843055e-18+3.36538467e-19j]
[ 4.96482289e+00+1.79605141e+02j  1.98938461e+01+4.43829634e+02j
 -2.33450171e-19+8.65246808e-19j]


In [24]:
print(Ha)
print(Hb)

[ 1.12142374e-18-5.02405974e-17j -1.61583160e-02+1.00956339e+00j
  1.56378568e-17-2.40840842e-18j]
[-3.28212882e-18-7.29124901e-17j  4.98802520e-02+1.31412760e+00j
  1.75137491e-17+1.84574463e-18j]
